# 9.2 Publication Text Analysis - step 2: prepare text corpus and lab-level data

This notebook:

1. Prepares the publication-level text corpus (title + abstract combined, one row per publication). It also translates text from German to English where appropriate.
2. Creates the lab-level subject-code benchmark shares and merges these benchmark shares with BL energy use and other control vars.
3. Creates a mapping from publication to labgroupids (one row per publication-labgroupid pair).

In [1]:
# Set up
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import langid
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [2]:
# Load data
publications = pd.read_csv(
    config.PUBLICATON_DATA / 
    "2_Processed" / 
    "publications_matched.csv"
)

labs = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Build the publication-lab mapping table

In [3]:
publications["pub_id"] = publications.index

pub_lab_map = publications[["pub_id", "matched_labgroupids"]].copy()
pub_lab_map["labgroupid"] = pub_lab_map["matched_labgroupids"].astype(str).str.split(";")
pub_lab_map = pub_lab_map.explode("labgroupid")
pub_lab_map["labgroupid"] = pub_lab_map["labgroupid"].str.strip()
pub_lab_map = pub_lab_map[["pub_id", "labgroupid"]]

print(f"Publications: {publications.shape[0]:,}")
print(f"Publication-lab mapping rows: {pub_lab_map.shape[0]:,}")
print(f"Distinct labs: {pub_lab_map['labgroupid'].nunique()}")

Publications: 8,014
Publication-lab mapping rows: 8,435
Distinct labs: 95


## (2) Combine title and description (abstract)

In [4]:
# Combine title and description into a single text field
def combine_text(row):
    parts = [row["title"]]
    if pd.notna(row["description"]):
        parts.append(row["description"])
    return " ".join(parts)

publications["raw_text"] = publications.apply(combine_text, axis=1)
#publications[["title", "description", "raw_text"]].head(3)

## (3) Deal with non-English language

In [5]:
# How many publications are tagged as non-English?
n_tagged_non_english = publications["language"].ne("eng").sum()
print(f"Publications where language is not English: {n_tagged_non_english:,}")

Publications where language is not English: 876


In [6]:
# Check raw_text when language is not "eng"
#publications.loc[publications["language"] != "eng", ["title", "description", "raw_text", "language"]].head(15)

The language column describes the language of the publication, not the title/abstract - we see that many "deu" entries have a title/abstract  in English. We use langid to find the language of the title/abstract (raw_text).

In [7]:
# Detect the actual language of raw_text
publications["langid_guess"] = publications["raw_text"].apply(lambda t: langid.classify(str(t))[0])
publications["is_non_english"] = publications["langid_guess"] != "en"

n_non_english = publications["is_non_english"].sum()
print(f"Publications detected as not English: {n_non_english:,} ({publications['is_non_english'].mean():.1%})")

Publications detected as not English: 584 (7.3%)


We see that there are fewer publications that are detected as non-English compared to where the publication has been tagged as non-English in the source data.

We check whether this is affecting all labs or whether there are a small number of labs with a high number of non-English pubs - this affects the severity of the language issue for the text analysis.

In [8]:
# Aggregate by labgroupid to get non-English publication stats
pub_lang_by_lab = pub_lab_map.merge(
    publications[["pub_id", "is_non_english"]], on="pub_id", how="left"
)

lab_lang_stats = pub_lang_by_lab.groupby("labgroupid").agg(
    n_pubs=("pub_id", "size"),
    n_non_english=("is_non_english", "sum"),
)
lab_lang_stats["share_non_english"] = lab_lang_stats["n_non_english"] / lab_lang_stats["n_pubs"]

# How many labs have at least 1 non-English publication?
n_labs_affected = (lab_lang_stats["n_non_english"] > 0).sum()
print(f"Labs with at least 1 non-English pub: {n_labs_affected} of {len(lab_lang_stats)}")
print("Distribution of each lab's share of non-English pubs:")
print(lab_lang_stats["share_non_english"].describe())

# Check what share of non-English pubs come from the top 10 labs
top10_share = (
    lab_lang_stats.sort_values("n_non_english", ascending=False).head(10)["n_non_english"].sum()
    / lab_lang_stats["n_non_english"].sum()
)
print(f"Share of all non-English pubs coming from just the top 10 labs: {top10_share:.1%}")

Labs with at least 1 non-English pub: 34 of 95
Distribution of each lab's share of non-English pubs:
count    95.000000
mean      0.040775
std       0.122393
min       0.000000
25%       0.000000
50%       0.000000
75%       0.013339
max       0.724138
Name: share_non_english, dtype: float64
Share of all non-English pubs coming from just the top 10 labs: 91.0%


We see that the language issues are concentrated in a few labs - this motivates translating raw_text to English.

In [9]:
# Check the language of the non-English publications
non_english_languages = publications.loc[publications["is_non_english"], "langid_guess"].value_counts()
print("Languages of non-English publications:")
print(non_english_languages.head(10))

Languages of non-English publications:
langid_guess
de    451
es     29
la     17
fr     16
it     16
ro      8
xh      6
id      6
pl      6
fi      5
Name: count, dtype: int64


In [10]:
# Print the raw_text of 5 randomly selected German publications
german_pubs = publications.loc[publications["langid_guess"] == "de", 
    "raw_text"].sample(5, random_state=42)
# print("Randomly selected German publications:")
# for i, text in enumerate(german_pubs, start=1):
    # print(text) # currently commented out for sensitivity

In [11]:
# Print the raw_text of 5 randomly selected Spanish publications
spanish_pubs = publications.loc[publications["langid_guess"] == "es", 
    "raw_text"].sample(5, random_state=42)
# print("Randomly selected Spanish publications:")
# for i, text in enumerate(spanish_pubs, start=1):
    # print(text) # currently commented out for sensitivity

In [12]:
# Print the raw_text of 5 randomly selected Latin publications
latin_pubs = publications.loc[publications["langid_guess"] == "la",
    "raw_text"].sample(5, random_state=42)
# print("Randomly selected Latin publications:")
# for i, text in enumerate(latin_pubs, start=1):
    # print(text) # currently commented out for sensitivity

We also see that most of the non-English texts are in German. We also see that the texts detected as German are in German while the Spanish texts are sometimes in English, and the Latin texts are all in English (typically with some Latin scientific terms or equations).
We therefore only translate the German texts to English.

In [13]:
# Translate German text to English (Helsinki-NLP/opus-mt-de-en via HF transformers)

# Load the model and tokenizer
MODEL_NAME = "Helsinki-NLP/opus-mt-de-en"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Texts to translate (unique German texts)
to_translate = publications.loc[publications["langid_guess"] == "de", "raw_text"]
unique_texts = to_translate.unique().tolist()
print(f"Translating {len(unique_texts)} unique German texts...")

# Translate in batches to avoid memory issues
# Set num_beans = 1 and max_length=512 to avoid memory issues and speed up translation
# Check every 5 batches to monitor progress
def translate_batch(texts, batch_size=8):
    outputs = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for batch_num, i in enumerate(range(0, len(texts), batch_size), start=1):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        translated = model.generate(**inputs, max_length=512, num_beams=1)
        outputs.extend(tokenizer.batch_decode(translated, skip_special_tokens=True))
        if batch_num % 5 == 0 or batch_num == n_batches:
            print(f"  batch {batch_num}/{n_batches} done ({len(outputs)}/{len(texts)} texts)")
    return outputs

translated_texts = translate_batch(unique_texts)
translation_map = dict(zip(unique_texts, translated_texts))

publications["translated_text"] = publications["raw_text"].map(translation_map)

# Greedy decoding (num_beams=1) occasionally produces an empty string instead
# of a real translation on short/unusually-punctuated inputs (a known failure
# mode of greedy decoding). An empty string isn't NaN, so it would silently
# slip past fillna() below and never fall back to raw_text - catch it explicitly.
publications["translated_text"] = publications["translated_text"].str.strip().replace("", pd.NA)

publications["text_final"] = publications["translated_text"].fillna(publications["raw_text"])

print(f"Rows translated: {publications['translated_text'].notna().sum()}")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translating 449 unique German texts...


  batch 5/57 done (40/449 texts)


  batch 10/57 done (80/449 texts)


  batch 15/57 done (120/449 texts)


...

  batch 40/57 done (320/449 texts)


  batch 45/57 done (360/449 texts)


  batch 50/57 done (400/449 texts)


  batch 55/57 done (440/449 texts)


  batch 57/57 done (449/449 texts)
Rows translated: 450


In [14]:
# Spot-check translation quality
sample_translated = publications.loc[publications["translated_text"].notna()].sample(5, random_state=0)
# for _, row in sample_translated.iterrows():
#     print("DE: ", row["raw_text"][:150])
#     print("EN: ", row["translated_text"][:150])
#     print() # commented out for sensitivity

The translations look reasonable.

## (4) Create subject benchmarks

In [15]:
# Extract classification codes from subject field
CODE_PATTERN = re.compile(r"^(\d{3})\s")

def extract_subject_codes(subject_str):
    if pd.isna(subject_str):
        return []
    tags = [t.strip() for t in subject_str.split("|") if t.strip()]
    return [m.group(1) for t in tags if (m := CODE_PATTERN.match(t))]

publications["subject_codes"] = publications["subject"].apply(extract_subject_codes)

In [16]:
# Create subject-code shares per lab (benchmark for text analysis).
# Join publication-level subject codes onto the pub-lab mapping, then explode
# each pub's list of codes - one row per (labgroupid, code)
pub_codes_by_lab = pub_lab_map.merge(
    publications[["pub_id", "subject_codes"]], on="pub_id", how="left"
)
codes_long = pub_codes_by_lab[["labgroupid", "subject_codes"]].explode("subject_codes")
codes_long = codes_long.dropna(subset=["subject_codes"])

# Count how many of each lab's publications carry each code...
code_counts = pd.crosstab(codes_long["labgroupid"], codes_long["subject_codes"])
# ...then divide by that lab's total publication count to get a share
n_pubs_per_lab_tmp = pub_lab_map.groupby("labgroupid").size()
subject_code_shares = code_counts.div(n_pubs_per_lab_tmp, axis=0).fillna(0)
subject_code_shares.columns = [f"subj_code_{c}" for c in subject_code_shares.columns]
subject_code_shares = subject_code_shares.reset_index()  # labgroupid back to a normal column, for merging

# print(f"Subject-code share columns built: {subject_code_shares.shape[1] - 1}")
# subject_code_shares.head(3)

## (5) Merge subject shares with lab-level data to get BL energy use and control vars

In [17]:
# Restrict to BL observations
bl_data = labs[labs["survey"] == "BL"].copy()

# Keep only energy and control vars cols
bl_data = bl_data[["labgroupid", "annual_electricity_total_mwh", "no_researchers", 
                   "faculty", "institute_id", "treated"]].copy()

In [18]:
# Merge publications with BL data
subject_code_shares["labgroupid"] = subject_code_shares["labgroupid"].astype(int)
bl_data["labgroupid"] = bl_data["labgroupid"].astype(int)
merged_df = pd.merge(bl_data, subject_code_shares, on="labgroupid", how="inner")

## (6) Save outputs

In [19]:
pub_out_cols = ["pub_id", "title", "description", "date", "n_matched_labgroupids",
                "raw_text", "langid_guess", "translated_text", "text_final"]
publications[pub_out_cols].to_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "publications_unique.csv",
    index=False)

pub_lab_map.to_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "pub_lab_mapping.csv",
    index=False)

merged_df.to_csv(
    config.PUBLICATON_DATA /
    "3_Clean" /
    "lab_level_subject_shares.csv", 
    index=False)